In [ ]:
!pip install pysbd

In [ ]:
import pysbd
import math
import os
import logging
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (AutoTokenizer, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, set_seed)
 
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Silence the benign ">512 tokens" warning: long docs are chopped into blocks
# below, so nothing over-length ever reaches the model.
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

CONFIGURATION

In [ ]:
Checkpoint = "roberta-large"
Out_directory = "/kaggle/working/dapt/roberta-large"
Block_size = 512
MLM_PROB = 0.15
Seed = 42

Loading in JSONL files

In [ ]:
Data_file_path = "/kaggle/input/datasets/lorenzouberti/unlabbled-fomc-data/*.jsonl"
raw_unlablled_jsonl = load_dataset("json", data_files={"train": Data_file_path})["train"]
raw_unlablled_jsonl = raw_unlablled_jsonl.filter(lambda x: isinstance(x["text"], str) and len(x["text"].strip()) > 0)
raw_unlablled_jsonl = raw_unlablled_jsonl.train_test_split(test_size=0.01, seed=Seed) 
raw = raw_unlablled_jsonl.copy()

In [ ]:
from datasets import Dataset, DatasetDict, concatenate_datasets

print(type(raw))   # plain dict = the culprit

if isinstance(raw, dict) and not isinstance(raw, DatasetDict):
    # hand-built dict (e.g. one entry per file): merge the values into one Dataset
    parts = [d["train"] if isinstance(d, DatasetDict) else d for d in raw.values()]
    raw = concatenate_datasets(parts)

if isinstance(raw, Dataset):
    # single merged Dataset: create the 1% perplexity holdout
    raw = raw.train_test_split(test_size=0.01, seed=Seed)

print(raw)   # should show DatasetDict({'train': ..., 'test': ...})

Tokenize

In [ ]:
def split_docs(batch):
    seg = pysbd.Segmenter(language="en", clean=False)
    out = []
    for doc in batch["text"]:
        sents = []
        for para in doc.split("\n\n"):   # paragraph pre-split (no-op if none present)
            para = para.strip()
            if para:
                sents.extend(s.strip() for s in seg.segment(para) if s.strip())
        out.append(sents)
    return {"sentences": out}
 
sent_ds = raw.map(split_docs, batched=True,
                  remove_columns=raw["train"].column_names,  # drops text AND any metadata
                  num_proc=4)
 
# Persist so you never pay the pysbd cost again:
sent_ds.save_to_disk("/kaggle/working/sentences")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(Checkpoint)

In [ ]:
CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
assert CLS_ID is not None and SEP_ID is not None, "checkpoint lacks CLS/SEP tokens"
BUDGET = Block_size - 2                     # room for the two special tokens
 
def pack(batch):
    docs = batch["sentences"]
    flat = [s for doc in docs for s in doc]  # batch-tokenize everything in one call
    enc = tokenizer(flat, add_special_tokens=False)["input_ids"]
 
    input_ids, special_masks = [], []
 
    def emit(cur):
        input_ids.append([CLS_ID] + cur + [SEP_ID])
        special_masks.append([1] + [0] * len(cur) + [1])   # 1 = never MLM-mask
 
    i = 0
    for doc in docs:
        cur = []
        for _ in doc:
            ids = enc[i][:BUDGET]   # hard-truncate rare monster "sentences" (tables etc.)
            i += 1
            if not ids:
                continue
            if len(cur) + len(ids) > BUDGET and cur:
                emit(cur)
                cur = []
            cur.extend(ids)
        if cur:                     # flush at doc end -> blocks never cross documents
            emit(cur)
    return {"input_ids": input_ids, "special_tokens_mask": special_masks}
 
lm_ds = sent_ds.map(pack, batched=True, remove_columns=["sentences"], num_proc=4)
print(lm_ds)   # ~30M tokens / ~512 per block ≈ 58k blocks

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(Checkpoint)
 
collator = DataCollatorForLanguageModeling(tokenizer, mlm=True,
                                           mlm_probability=MLM_PROB,
                                           pad_to_multiple_of=8)  # tensor-core friendly for fp16
 
args = TrainingArguments(
    output_dir=Out_directory,
    eval_strategy="steps",
    eval_steps=150,                    # only ~1080 total steps now, so eval finer
    save_strategy="steps",
    save_steps=150,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    learning_rate=3e-5,                # large models want a gentler rate (same lesson as the fine-tune)
    per_device_train_batch_size=4,     # large at block 512 won't fit 8/GPU on a 15GB T4
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,    # 4 x 2 GPUs x 16 = effective 128, identical to the base run
    # gradient_checkpointing=True,     # ONLY if OOM; costs ~35% speed -> then also set max_steps~700
    num_train_epochs=3,                # see timing note below
    # max_steps=950,                   # optional harder safety cap (~9h); overrides epochs
    warmup_ratio=0.06,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=Seed,
)
 
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_ds["train"],
    eval_dataset=lm_ds["test"],
    processing_class=tokenizer,
    data_collator=collator,
)
 
# %%
trainer.train()
 
eval_out = trainer.evaluate()
print(f"eval loss {eval_out['eval_loss']:.4f} | perplexity {math.exp(eval_out['eval_loss']):.2f}")
 
# %%
final_dir = f"{Out_directory}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))   # attach the secret to THIS notebook too

model.push_to_hub("LorenzoAleCon29/roberta-large-fomc-dapt", private=True)
tokenizer.push_to_hub("LorenzoAleCon29/roberta-large-fomc-dapt", private=True)